# Reproducible CatBoost validation run

Обучение только на `train.csv`, проверка только на `val.csv`. Конфигурация фиксирует `task_type="CPU"`, `thread_count=1`, `random_seed=42`. `holdout.csv` здесь не используется.

In [2]:
%pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [1]:
# =========================
# Reproducible CatBoost validation run
# train.csv -> train only
# val.csv   -> validation only
# holdout.csv is NOT used here
# =========================

import pandas as pd
import numpy as np

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_percentage_error

# =========================
# 0. Global reproducibility
# =========================

SEED = 42
np.random.seed(SEED)

TRAIN_PATH = 'train.csv'
VAL_PATH = 'val.csv'
TARGET_COL = 'target_2'
ID_COL = 'ID'

# =========================
# 1. Load data
# =========================

train = pd.read_csv(TRAIN_PATH)
val = pd.read_csv(VAL_PATH)

# Remove accidental spaces in column names, e.g. ' target_2 ' -> 'target_2'
for df in [train, val]:
    df.columns = df.columns.str.strip()

assert TARGET_COL in train.columns, f'{TARGET_COL} not found in train columns'
assert TARGET_COL in val.columns, f'{TARGET_COL} not found in val columns'

# =========================
# 2. Feature engineering
# =========================

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['DIST_TO_ADM_CENTER'] = np.log1p(df['DIST_TO_ADM_CENTER'])

    df['TOTAL_RANK_COMPS_CANNIBALS'] = (
        df['TOTAL_RANK_COMPS_CANNIBALS']
        .fillna(0)
        .astype(np.int64)
    )

    df['families_per_competitor'] = (
        df['HuffFamilies'] / (df['TOTAL_RANK_COMPS_CANNIBALS'] + 1)
    )

    df['attraction_per_square'] = (
        df['TRADING_PATCH_SCORE'] / (df['TRADE_SQUARE'] + 1e-6)
    )

    df['families_x_24h'] = (
        df['HuffFamilies'] * df['ENTIRE_DAY']
    )

    df['traffic_x_24h'] = (
        df['ROUTES_CNT'] * df['ENTIRE_DAY']
    )

    df['families_x_attraction'] = (
        df['HuffFamilies'] * df['TRADING_PATCH_SCORE']
    )

    return df

train = add_features(train)
val = add_features(val)

# =========================
# 3. Feature lists
# =========================

categorical = [
    'REGION', 'BRANCH', 'CITY', 'subject', 'SUBFRMT', 'HOSPITAL',
    'KINDERGARTENS', 'BANKS', 'MK_ON_MM', 'CROSSWALK',
    'CROSSROAD', 'DENSITY_FAMILY_TYPE', 'COLLEGES',
    'IN_COAL_CITY', 'IN_OIL_CITY', 'CITY_TYPE', 'ON_DUPLICATE_ROAD',
    'INSIDE_YARD', 'ON_INSIDE_DISTRICT_ROAD', 'ON_MAIN_CITY_ROAD',
    'ON_INTERCITY_HIGHWAY', 'ENTRANCE_TO_DISTRICT', 'ON_DISTRICT_BOTTOM',
    'HUB', 'ALCOHOL', 'TOBACCO', 'LOCATION_MARKET', 'STATIONS', 'TRC',
    'METRO', 'LOCATION_PARK', 'MINI_TRC', 'TRADING_PATCH', 'ENTIRE_DAY',
    'MORNING_ROADSIDE', 'SEA', 'FLOORS_TZ', 'SNT', 'LOCATION_TYPE',
    'new_buildings', 'change_CA'
]

base_numeric = [
    'TRADE_SQUARE',
    'HuffFamilies',
    'HuffRelativeFamilies',
    'ROUTES_CNT',
    'huff_hotel',
    'TRADING_PATCH_SCORE',
    'RENT_HEX',
    'HUFF_RANK_COMPS_CANNIBALS',
    'TOTAL_RANK_COMPS_CANNIBALS',
    'DIST_TO_ADM_CENTER',
    'PARKING',
    'month_count'
]

best_engineered = [
    'families_per_competitor',
    'attraction_per_square',
    'families_x_24h',
    'traffic_x_24h',
    'families_x_attraction'
]

drop_cols = [ID_COL, 'target_1', TARGET_COL, '']

features = [
    c for c in base_numeric + categorical + best_engineered
    if c in train.columns and c in val.columns and c not in drop_cols
]
features = list(dict.fromkeys(features))

cat_features = [
    c for c in categorical
    if c in features
]

print('n_features:', len(features))
print('cat_features:', cat_features)

# =========================
# 4. Target transform
# =========================

y_train_log = np.log1p(train[TARGET_COL])
y_val = val[TARGET_COL].copy()
y_val_log = np.log1p(y_val)

# =========================
# 5. Best fixed config from the validation notebook
#    + deterministic inference/training settings
# =========================

best_params_found = {
    'w_trcc': 1.445121247089796,
    'w_sub': 0.8245213497148861,
    'w_mc': 0.19477216208456857,
    'w_hf': 1.7124606550499168,
    'w_ts': 0.4538034002556876,
    'w_subj': 1.5129601330685374,
    'w_hrf': 1.1819611157732426,
    'alpha': 0.22565922419540407,
    'depth': 5,
    'learning_rate': 0.04981885495006629,
    'l2_leaf_reg': 2.1076625903601975,
    'random_strength': 8.053316941452,
    'rsm': 0.7598914863088294,
    'subsample': 0.7400675626191421,
}

feature_weight_map = {
    'TOTAL_RANK_COMPS_CANNIBALS': best_params_found['w_trcc'],
    'SUBFRMT': best_params_found['w_sub'],
    'month_count': best_params_found['w_mc'],
    'HuffFamilies': best_params_found['w_hf'],
    'TRADE_SQUARE': best_params_found['w_ts'],
    'subject': best_params_found['w_subj'],
    'HuffRelativeFamilies': best_params_found['w_hrf'],
}

# CatBoost expects feature_weights aligned with the feature order.
feature_weights = [feature_weight_map.get(col, 1.0) for col in features]

params = {
    'loss_function': f"Quantile:alpha={best_params_found['alpha']}",
    'depth': best_params_found['depth'],
    'learning_rate': best_params_found['learning_rate'],
    'iterations': 4000,
    'l2_leaf_reg': best_params_found['l2_leaf_reg'],
    'random_strength': best_params_found['random_strength'],
    'rsm': best_params_found['rsm'],
    'bootstrap_type': 'Bernoulli',
    'subsample': best_params_found['subsample'],
    'eval_metric': 'MAPE',
    'random_seed': SEED,
    'task_type': 'CPU',
    'thread_count': 1,
    'feature_weights': feature_weights,
    'allow_writing_files': False,
    'verbose': 100,
}

# =========================
# 6. Train on train, validate on val
# =========================

train_pool = Pool(
    train[features],
    y_train_log,
    cat_features=cat_features
)

val_pool = Pool(
    val[features],
    y_val_log,
    cat_features=cat_features
)

model = CatBoostRegressor(**params)

model.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=200,
    use_best_model=True
)

# =========================
# 7. Validation predictions and metric in original target scale
# =========================

val_preds = np.expm1(model.predict(val[features]))
val_preds = np.maximum(val_preds, 0)

val_mape = mean_absolute_percentage_error(y_val, val_preds)

print('\nFINAL HONEST VALIDATION MAPE:', val_mape)
print('FINAL HONEST VALIDATION MAPE, %:', val_mape * 100)
print('best_iteration:', model.get_best_iteration())

n_features: 58
cat_features: ['REGION', 'BRANCH', 'CITY', 'subject', 'SUBFRMT', 'HOSPITAL', 'KINDERGARTENS', 'BANKS', 'MK_ON_MM', 'CROSSWALK', 'CROSSROAD', 'DENSITY_FAMILY_TYPE', 'COLLEGES', 'IN_COAL_CITY', 'IN_OIL_CITY', 'CITY_TYPE', 'ON_DUPLICATE_ROAD', 'INSIDE_YARD', 'ON_INSIDE_DISTRICT_ROAD', 'ON_MAIN_CITY_ROAD', 'ON_INTERCITY_HIGHWAY', 'ENTRANCE_TO_DISTRICT', 'ON_DISTRICT_BOTTOM', 'HUB', 'ALCOHOL', 'TOBACCO', 'LOCATION_MARKET', 'STATIONS', 'TRC', 'METRO', 'LOCATION_PARK', 'MINI_TRC', 'TRADING_PATCH', 'ENTIRE_DAY', 'MORNING_ROADSIDE', 'SEA', 'FLOORS_TZ', 'SNT', 'LOCATION_TYPE', 'new_buildings', 'change_CA']
0:	learn: 0.0221302	test: 0.0180210	best: 0.0180210 (0)	total: 72.5ms	remaining: 4m 49s
100:	learn: 0.0158060	test: 0.0137062	best: 0.0137062 (100)	total: 1.32s	remaining: 51s
200:	learn: 0.0148980	test: 0.0132766	best: 0.0132766 (200)	total: 2.5s	remaining: 47.3s
300:	learn: 0.0143934	test: 0.0129333	best: 0.0129333 (300)	total: 3.85s	remaining: 47.3s
400:	learn: 0.0141236	test

Starting seed averaging ensemble with 7 models...


NameError: name 'holdout' is not defined

запустим еще раз, чтобы убедиться в воспроизводимости (MAPE должен быть таким же, как в прошлом запуске)

In [3]:
model = CatBoostRegressor(**params)

model.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=200,
    use_best_model=True
)

# =========================
# 7. Validation predictions and metric in original target scale
# =========================

val_preds = np.expm1(model.predict(val[features]))
val_preds = np.maximum(val_preds, 0)

val_mape = mean_absolute_percentage_error(y_val, val_preds)

print('\nFINAL HONEST VALIDATION MAPE:', val_mape)
print('FINAL HONEST VALIDATION MAPE, %:', val_mape * 100)
print('best_iteration:', model.get_best_iteration())

0:	learn: 0.0221302	test: 0.0180210	best: 0.0180210 (0)	total: 17ms	remaining: 1m 7s
100:	learn: 0.0158060	test: 0.0137062	best: 0.0137062 (100)	total: 1.24s	remaining: 47.9s
200:	learn: 0.0148980	test: 0.0132766	best: 0.0132766 (200)	total: 2.45s	remaining: 46.3s
300:	learn: 0.0143934	test: 0.0129333	best: 0.0129333 (300)	total: 3.83s	remaining: 47s
400:	learn: 0.0141236	test: 0.0127916	best: 0.0127916 (400)	total: 5.22s	remaining: 46.9s
500:	learn: 0.0139389	test: 0.0127127	best: 0.0127127 (500)	total: 6.66s	remaining: 46.5s
600:	learn: 0.0137845	test: 0.0126501	best: 0.0126501 (600)	total: 8.15s	remaining: 46.1s
700:	learn: 0.0136455	test: 0.0126003	best: 0.0126001 (699)	total: 9.63s	remaining: 45.3s
800:	learn: 0.0135099	test: 0.0125621	best: 0.0125621 (800)	total: 11.2s	remaining: 44.6s
900:	learn: 0.0134090	test: 0.0125316	best: 0.0125316 (900)	total: 12.7s	remaining: 43.6s
1000:	learn: 0.0133085	test: 0.0125070	best: 0.0125006 (986)	total: 14.2s	remaining: 42.5s
1100:	learn: 0.0

MAPE такой же

# Submission save

In [6]:
# =========================
# 8. Train final model on train + val, predict holdout
# =========================

HOLDOUT_PATH = 'holdout.csv'

holdout = pd.read_csv(HOLDOUT_PATH)
holdout.columns = holdout.columns.str.strip()

holdout = add_features(holdout)

train_full = pd.concat([train, val], axis=0).reset_index(drop=True)
y_full_log = np.log1p(train_full[TARGET_COL])

final_pool = Pool(
    train_full[features],
    y_full_log,
    cat_features=cat_features
)

final_model = CatBoostRegressor(**params)

final_model.fit(final_pool)

holdout_preds = np.expm1(
    final_model.predict(holdout[features])
)

holdout_preds = np.maximum(holdout_preds, 0)

# =========================
# 9. Save submission
# =========================

submission = pd.DataFrame({
    'ID': holdout[ID_COL].values,
    'PREDICT': holdout_preds
})

assert submission['PREDICT'].isna().sum() == 0
assert (submission['PREDICT'] >= 0).all()
assert len(submission) == len(holdout)

submission.to_csv(
    'predictions.csv',
    index=False,
    encoding='utf-8'
)

submission.head()

0:	learn: 0.0222362	total: 128ms	remaining: 8m 33s
100:	learn: 0.0160191	total: 9.42s	remaining: 6m 3s
200:	learn: 0.0151123	total: 12.8s	remaining: 4m 2s
300:	learn: 0.0145030	total: 17s	remaining: 3m 28s
400:	learn: 0.0142279	total: 21.7s	remaining: 3m 14s
500:	learn: 0.0140057	total: 25.6s	remaining: 2m 58s
600:	learn: 0.0138494	total: 29.9s	remaining: 2m 48s
700:	learn: 0.0137177	total: 41.5s	remaining: 3m 15s
800:	learn: 0.0136052	total: 47.2s	remaining: 3m 8s
900:	learn: 0.0135207	total: 51.9s	remaining: 2m 58s
1000:	learn: 0.0134365	total: 55.9s	remaining: 2m 47s
1100:	learn: 0.0133741	total: 1m	remaining: 2m 40s
1200:	learn: 0.0132977	total: 1m 4s	remaining: 2m 31s
1300:	learn: 0.0132433	total: 1m 8s	remaining: 2m 22s
1400:	learn: 0.0131758	total: 1m 13s	remaining: 2m 16s
1500:	learn: 0.0131214	total: 1m 17s	remaining: 2m 9s
1600:	learn: 0.0130736	total: 1m 21s	remaining: 2m 2s
1700:	learn: 0.0130221	total: 1m 26s	remaining: 1m 57s
1800:	learn: 0.0129643	total: 1m 30s	remaining

,ID,PREDICT
0,17178,171544.391761
1,17179,218531.572794
2,17180,185457.594094
3,17181,285344.760909
4,17182,184011.134925


In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_percentage_error

# 1. Список из 10 сидов
seeds = [42, 1, 10, 555, 777, 2024, 99, 123, 8, 21]

val_predictions_list = []
holdout_predictions_list = []

# Гарантируем, что holdout загружен и обработан
holdout = pd.read_csv('holdout.csv')
holdout.columns = holdout.columns.str.strip()
holdout = add_features(holdout)

print(f"Starting Seed Averaging Ensemble ({len(seeds)} models) for validation check...")

for i, s in enumerate(seeds):
    # Копируем параметры и меняем сид
    curr_params = params.copy()
    curr_params['random_seed'] = s
    
    # Обучаем СТРОГО на train, чтобы честно проверить на val
    model = CatBoostRegressor(**curr_params)
    model.fit(
        Pool(train[features], y_train_log, cat_features=cat_features),
        eval_set=Pool(val[features], y_val_log, cat_features=cat_features),
        early_stopping_rounds=150,
        use_best_model=True,
        verbose=False
    )
    
    # Предсказание для VAL (для замера метрики)
    v_preds = np.maximum(np.expm1(model.predict(val[features])), 0)
    val_predictions_list.append(v_preds)
    
    # Предсказание для HOLDOUT (для финального файла)
    h_preds = np.maximum(np.expm1(model.predict(holdout[features])), 0)
    holdout_predictions_list.append(h_preds)
    
    # Точечный MAPE текущей модели (для логов)
    current_mape = mean_absolute_percentage_error(y_val, v_preds)
    print(f"Model {i+1}/{len(seeds)} | Seed {s} | Iterations: {model.get_best_iteration()} | MAPE: {current_mape:.5f}")

# --- РАСЧЕТ АНСАМБЛЕВОГО КАЧЕСТВА ---

# 1. Усредняем все предсказания для валидации
avg_val_preds = np.mean(val_predictions_list, axis=0)
ensemble_mape = mean_absolute_percentage_error(y_val, avg_val_preds)

# 2. Считаем средний MAPE одиночных моделей для сравнения
individual_mapes = [mean_absolute_percentage_error(y_val, p) for p in val_predictions_list]
mean_individual_mape = np.mean(individual_mapes)

print("\n" + "="*40)
print(f"AVG SINGLE MODEL MAPE: {mean_individual_mape:.5f}")
print(f"ENSEMBLE MAPE (10 SEEDS): {ensemble_mape:.5f}")
print(f"ABSOLUTE IMPROVEMENT: {mean_individual_mape - ensemble_mape:.5f}")
print("="*40)


Starting Seed Averaging Ensemble (10 models) for validation check...
Model 1/10 | Seed 42 | Iterations: 1892 | MAPE: 0.14859
Model 2/10 | Seed 1 | Iterations: 1822 | MAPE: 0.15037
Model 3/10 | Seed 10 | Iterations: 1915 | MAPE: 0.14919
Model 4/10 | Seed 555 | Iterations: 2591 | MAPE: 0.15035
Model 5/10 | Seed 777 | Iterations: 2311 | MAPE: 0.14986
Model 6/10 | Seed 2024 | Iterations: 2188 | MAPE: 0.14981
Model 7/10 | Seed 99 | Iterations: 2380 | MAPE: 0.14956
Model 8/10 | Seed 123 | Iterations: 1274 | MAPE: 0.15045
Model 9/10 | Seed 8 | Iterations: 2129 | MAPE: 0.14943
Model 10/10 | Seed 21 | Iterations: 2696 | MAPE: 0.14988



NameError: name 'params_found' is not defined

# АНСАМБЛЬ

In [10]:
# Список сидов для ансамбля (5-10 штук достаточно)
seeds = [42, 1, 10, 555, 777, 2024, 99, 123, 8, 21]

holdout_predictions = []
val_predictions_list = []

# Подготовка полных данных для финального обучения
train_full = pd.concat([train, val], axis=0).reset_index(drop=True)
y_full_log = np.log1p(train_full[TARGET_COL])

print(f"Starting seed averaging ensemble with {len(seeds)} models...")
HOLDOUT_PATH = 'holdout.csv'
holdout = pd.read_csv(HOLDOUT_PATH)
holdout.columns = holdout.columns.str.strip()
holdout = add_features(holdout)
for i, s in enumerate(seeds):
    params['random_seed'] = s
    
    # 1. Обучаем модель на всем доступном объеме (train+val)
    # Используем количество итераций из вашего лучшего запуска (например, best_iteration)
    # Или обучаем до конца, так как на полном наборе нет val_set
    model = CatBoostRegressor(**params)
    model.fit(Pool(train_full[features], y_full_log, cat_features=cat_features), verbose=False)
    
    # 2. Предсказываем для holdout (в логарифмах)
    holdout_pred_log = model.predict(holdout[features])
    
    # 3. Переводим в реальный масштаб и сохраняем
    holdout_pred_real = np.maximum(np.expm1(holdout_pred_log), 0)
    holdout_predictions.append(holdout_pred_real)
    
    print(f"Model {i+1}/{len(seeds)} (seed {s}) trained.")

# 4. Финальное усреднение (простое среднее арифметическое)
final_holdout_preds = np.mean(holdout_predictions, axis=0)

# 5. Сохранение финального результата
submission = pd.DataFrame({
    'id': holdout[ID_COL].values,
    'PREDICT': final_holdout_preds
})
submission.to_csv('predictions_ensemble.csv', index=False)

print("\nEnsemble complete! Result saved to predictions_ensemble.csv")

Starting seed averaging ensemble with 10 models...
Model 1/10 (seed 42) trained.
Model 2/10 (seed 1) trained.
Model 3/10 (seed 10) trained.
Model 4/10 (seed 555) trained.
Model 5/10 (seed 777) trained.
Model 6/10 (seed 2024) trained.
Model 7/10 (seed 99) trained.
Model 8/10 (seed 123) trained.
Model 9/10 (seed 8) trained.
Model 10/10 (seed 21) trained.

Ensemble complete! Result saved to predictions_ensemble.csv


# АНСАМБЛЬ С СОХРАНЕНИЕМ ВЕСОВ


In [12]:
import os
import json

os.makedirs('models', exist_ok=True)

seeds = [42, 1, 10, 555, 777, 2024, 99, 123, 8, 21]
holdout_predictions = []

train_full = pd.concat([train, val], axis=0).reset_index(drop=True)
y_full_log = np.log1p(train_full[TARGET_COL])

metadata = {
    'features': features,
    'cat_features': cat_features,
    'best_params_found': best_params_found
}
with open('models/metadata.json', 'w') as f:
    json.dump(metadata, f)

print(f"Starting seed averaging ensemble and saving models...")

for i, s in enumerate(seeds):
    current_params = params.copy()
    current_params['random_seed'] = s
    
    model = CatBoostRegressor(**current_params)
    model.fit(Pool(train_full[features], y_full_log, cat_features=cat_features), verbose=False)
    
    model_path = f'models/catboost_seed_{s}.cbm'
    model.save_model(model_path)
    
    h_preds = np.maximum(np.expm1(model.predict(holdout[features])), 0)
    holdout_predictions.append(h_preds)
    
    print(f"Model {i+1}/{len(seeds)} (seed {s}) saved to {model_path}")

final_holdout_preds = np.mean(holdout_predictions, axis=0)
submission = pd.DataFrame({'id': holdout[ID_COL].values, 'PREDICT': final_holdout_preds})
submission.to_csv('predictions_ensemble.csv', index=False)

Starting seed averaging ensemble and saving models...
Model 1/10 (seed 42) saved to models/catboost_seed_42.cbm
Model 2/10 (seed 1) saved to models/catboost_seed_1.cbm
Model 3/10 (seed 10) saved to models/catboost_seed_10.cbm
Model 4/10 (seed 555) saved to models/catboost_seed_555.cbm
Model 5/10 (seed 777) saved to models/catboost_seed_777.cbm
Model 6/10 (seed 2024) saved to models/catboost_seed_2024.cbm
Model 7/10 (seed 99) saved to models/catboost_seed_99.cbm
Model 8/10 (seed 123) saved to models/catboost_seed_123.cbm
Model 9/10 (seed 8) saved to models/catboost_seed_8.cbm
Model 10/10 (seed 21) saved to models/catboost_seed_21.cbm


# ЗАГРУЗКА МОДЕЛЕЙ И ПРОТОТИП ИНФЕРЕНСА

In [16]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import json
import glob

def add_features_inference(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df['DIST_TO_ADM_CENTER'] = np.log1p(df['DIST_TO_ADM_CENTER'].fillna(0))
    df['TOTAL_RANK_COMPS_CANNIBALS'] = df['TOTAL_RANK_COMPS_CANNIBALS'].fillna(0).astype(np.int64)
    df['families_per_competitor'] = df['HuffFamilies'] / (df['TOTAL_RANK_COMPS_CANNIBALS'] + 1)
    df['attraction_per_square'] = df['TRADING_PATCH_SCORE'] / (df['TRADE_SQUARE'] + 1e-6)
    df['families_x_24h'] = df['HuffFamilies'] * df['ENTIRE_DAY']
    df['traffic_x_24h'] = df['ROUTES_CNT'] * df['ENTIRE_DAY']
    df['families_x_attraction'] = df['HuffFamilies'] * df['TRADING_PATCH_SCORE']
    return df

def run_ensemble_inference(input_csv_path, models_dir='models'):
    with open(f'{models_dir}/metadata.json', 'r') as f:
        meta = json.load(f)
    
    features = meta['features']
    
    raw_df = pd.read_csv(input_csv_path)
    df_proc = add_features_inference(raw_df)
    
    model_files = glob.glob(f'{models_dir}/*.cbm')
    ensemble_preds = []
    
    print(f"Loading {len(model_files)} models for inference...")
    
    for model_path in model_files:
        model = CatBoostRegressor()
        model.load_model(model_path)
        
        preds_log = model.predict(df_proc[features])
        preds_real = np.maximum(np.expm1(preds_log), 0)
        ensemble_preds.append(preds_real)
        
    final_preds = np.mean(ensemble_preds, axis=0)
    
    result = pd.DataFrame({
        'id': raw_df['ID'].values,
        'PREDICT': final_preds
    })
    
    return result

# Пример вызова:
submission_df = run_ensemble_inference('holdout.csv')
submission_df.to_csv('final_submission.csv', index=False)

df1 = submission_df
df2 = pd.read_csv('predictions_ensemble_final.csv')
numeric_equal = np.allclose(df1['PREDICT'], df2['PREDICT'], atol=1e-7)
print(f"Числовые значения совпадают: {numeric_equal}")

Loading 10 models for inference...
Числовые значения совпадают: True
